# One-Hot Encoding (OHE) for Text: Implementation & Failure Modes

One-Hot Encoding represents discrete tokens as binary vectors of length $|V|$ (vocabulary size), where only the token's assigned index is marked with `1` and all other entries are `0`.

### Core Objectives:
1. Build a pure Python token-to-vector encoder from scratch.
2. Inspect sequence matrix representations for varying document lengths.
3. Empirically verify the 4 fatal failure modes: **Non-uniform shapes**, **Sparsity**, **Orthogonality (Zero semantic similarity)**, and **Out-Of-Vocabulary (OOV) collapse**.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

## 1. Defining the Corpus & Extracting the Vocabulary

In [ ]:
corpus = [
    "The food is good",
    "The food is bad",
    "Pizza is amazing"
]

# Tokenize, lowercase, and construct distinct vocabulary
tokenized_docs = [doc.lower().split() for doc in corpus]
vocabulary = sorted(list(set(token for doc in tokenized_docs for token in doc)))
word_to_idx = {word: idx for idx, word in enumerate(vocabulary)}

print(f"Corpus size: {len(corpus)} documents")
print(f"Vocabulary: {vocabulary}")
print(f"Vocabulary Size: {len(vocabulary)}")
print(f"Word-to-Index Lookup Table: {word_to_idx}")

## 2. Pure Python One-Hot Encoder from Scratch

In [ ]:
def encode_word_one_hot(word, vocab_map):
    """Generates a binary vector of dimension |V| for a given token."""
    vector = np.zeros(len(vocab_map), dtype=int)
    if word in vocab_map:
        vector[vocab_map[word]] = 1
    return vector

# Demonstrate encoding on sample tokens
v_food = encode_word_one_hot("food", word_to_idx)
v_pizza = encode_word_one_hot("pizza", word_to_idx)

print(f"v('food'):  {v_food}")
print(f"v('pizza'): {v_pizza}")

## 3. Sequence Matrix Representation of Documents

When whole sentences are represented using word-level one-hot vectors, each document becomes a 2D matrix of shape `(sequence_length, |V|)`. Notice how variable sentence lengths produce incompatible matrix dimensions.

In [ ]:
# Document 1: "The food is good" (4 tokens) -> Shape (4, 7)
doc1_matrix = np.array([encode_word_one_hot(w, word_to_idx) for w in tokenized_docs[0]])
df_doc1 = pd.DataFrame(doc1_matrix, index=tokenized_docs[0], columns=vocabulary)
print("Document 1 Matrix Shape:", doc1_matrix.shape)
display(df_doc1)

# Document 3: "Pizza is amazing" (3 tokens) -> Shape (3, 7)
doc3_matrix = np.array([encode_word_one_hot(w, word_to_idx) for w in tokenized_docs[2]])
print("Document 3 Matrix Shape:", doc3_matrix.shape)
print("Problem: Standard ML models cannot handle variable rows (4 vs 3) across samples!")

## 4. The Orthogonality & Semantic Blindness Problem

In one-hot vector space, the dot product between any two distinct word vectors is strictly zero. Cosine similarity is always 0.0, rendering synonyms completely disconnected.

In [ ]:
v_food = encode_word_one_hot("food", word_to_idx).reshape(1, -1)
v_pizza = encode_word_one_hot("pizza", word_to_idx).reshape(1, -1)
v_bad = encode_word_one_hot("bad", word_to_idx).reshape(1, -1)

sim_food_pizza = cosine_similarity(v_food, v_pizza)[0][0]
sim_food_bad = cosine_similarity(v_food, v_bad)[0][0]

print(f"Cosine Similarity('food', 'pizza'): {sim_food_pizza:.4f}")
print(f"Cosine Similarity('food', 'bad')  : {sim_food_bad:.4f}")
print(f"Dot Product('food', 'pizza')      : {np.dot(v_food, v_pizza.T)[0][0]}")
print("Conclusion: One-Hot Encoding cannot determine that 'food' is closer to 'pizza' than to 'bad'.")

## 5. Out-of-Vocabulary (OOV) Collapse

If an unseen word appears at test time, it has no index in the vocabulary and silently collapses to an all-zero vector.

In [ ]:
test_token = "burger"
v_test = encode_word_one_hot(test_token, word_to_idx)

print(f"Vector for unseen word '{test_token}':", v_test)
print(f"Is vector completely empty (all zeros)? {np.all(v_test == 0)}")

## 6. Document-Level Scikit-Learn Equivalent (Binary Bag of Words)

To produce fixed-dimension document vectors from binary indicators, Scikit-Learn uses `CountVectorizer(binary=True)`. Each row represents an entire document as a single vector of length $|V|$.

In [ ]:
cv_binary = CountVectorizer(binary=True)
X_bin = cv_binary.fit_transform(corpus)

df_bin = pd.DataFrame(
    X_bin.toarray(),
    columns=cv_binary.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("Fixed Document Matrix Shape:", X_bin.shape)
display(df_bin)